# Data Pipeline for LLM Pretraining

> Build a complete data preprocessing pipeline from raw text to training-ready format

**Pipeline Steps:**
1. 📥 Data Collection (download/load)
2. 🧹 Text Cleaning (deduplication, filtering)
3. 🔤 Tokenization
4. 📊 Chunking & Batching
5. 💾 DataLoader Creation

In [ ]:
import os
import re
import json
import hashlib
import random
from collections import Counter
from typing import List, Dict, Tuple, Iterator

import numpy as np
import matplotlib.pyplot as plt

## 1. Sample Dataset Creation

In real scenarios, you'd load from: Wikipedia, Common Crawl, GitHub, Books, etc.

In [ ]:
# Create a sample dataset
sample_dataset = [
    """Machine learning is a subset of artificial intelligence that enables systems to learn from data.
    Deep learning uses neural networks with many layers to model complex patterns.
    Natural language processing allows computers to understand human language.""",
    
    """Python is a versatile programming language used in data science and web development.
    Python supports multiple paradigms including object-oriented and functional programming.
    Many developers prefer Python for its readability and extensive libraries.""",
    
    """The transformer architecture revolutionized natural language processing in 2017.
    Attention mechanisms allow models to focus on relevant parts of the input sequence.
    GPT models use decoder-only transformers for text generation tasks.""",
    
    """Machine learning is a subset of artificial intelligence that enables systems to learn from data.
    This is a duplicate sentence to test deduplication.
    Machine learning is a subset of artificial intelligence that enables systems to learn from data.""",  # Duplicate
    
    """   
    
    
    This document has excessive whitespace and empty lines.
    
    
       """,  # Bad formatting
    
    """Contact us at spam@example.com or visit http://malicious-site.com
    Buy cheap products now!!! Click here!!!
    Special offer: $$$$ FREE $$$$ money!!!""",  # Spam/low quality
    
    """The quick brown fox jumps over the lazy dog.
    The quick brown fox jumps over the lazy dog.
    The quick brown fox jumps over the lazy dog.""",  # Repetitive
]

print(f"Total documents: {len(sample_dataset)}")
print(f"Total characters: {sum(len(d) for d in sample_dataset)}")

## 2. Text Cleaning Pipeline

In [ ]:
class TextCleaner:
    """Complete text cleaning pipeline"""
    
    def __init__(self, min_length=50, max_length=10000):
        self.min_length = min_length
        self.max_length = max_length
    
    def normalize_whitespace(self, text: str) -> str:
        """Remove extra whitespace and normalize"""
        # Replace multiple spaces/newlines with single
        text = re.sub(r'\s+', ' ', text)
        # Remove leading/trailing whitespace
        text = text.strip()
        return text
    
    def remove_urls_emails(self, text: str) -> str:
        """Remove URLs and email addresses"""
        # Remove URLs
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        # Remove emails
        text = re.sub(r'\S+@\S+', '', text)
        return text
    
    def remove_excessive_punctuation(self, text: str) -> str:
        """Remove excessive punctuation and spam patterns"""
        # Replace 3+ consecutive punctuation with single
        text = re.sub(r'[!?]{2,}', '!', text)
        text = re.sub(r'\.{3,}', '...', text)
        # Remove excessive dollar signs
        text = re.sub(r'\${2,}', '', text)
        return text
    
    def filter_length(self, text: str) -> bool:
        """Check if text meets length requirements"""
        return self.min_length <= len(text) <= self.max_length
    
    def calculate_quality_score(self, text: str) -> float:
        """Calculate text quality score (0-1)"""
        if not text:
            return 0.0
        
        score = 1.0
        
        # Penalize excessive uppercase
        upper_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)
        if upper_ratio > 0.5:
            score -= 0.3
        
        # Penalize excessive punctuation
        punct_ratio = sum(1 for c in text if c in '!?@#$%^&*') / max(len(text), 1)
        if punct_ratio > 0.1:
            score -= 0.3
        
        # Penalize very short words (spam indicator)
        words = text.split()
        avg_word_len = np.mean([len(w) for w in words]) if words else 0
        if avg_word_len < 3:
            score -= 0.2
        
        return max(score, 0.0)
    
    def clean(self, text: str) -> Tuple[str, Dict]:
        """Full cleaning pipeline"""
        metadata = {'original_length': len(text)}
        
        # Step 1: Normalize whitespace
        text = self.normalize_whitespace(text)
        
        # Step 2: Remove URLs and emails
        text = self.remove_urls_emails(text)
        
        # Step 3: Remove excessive punctuation
        text = self.remove_excessive_punctuation(text)
        
        # Step 4: Normalize whitespace again
        text = self.normalize_whitespace(text)
        
        # Step 5: Quality check
        quality = self.calculate_quality_score(text)
        metadata['quality_score'] = quality
        metadata['clean_length'] = len(text)
        
        # Step 6: Length filter
        if not self.filter_length(text):
            metadata['rejected'] = True
            metadata['reason'] = 'length'
            return '', metadata
        
        if quality < 0.5:
            metadata['rejected'] = True
            metadata['reason'] = 'quality'
            return '', metadata
        
        metadata['rejected'] = False
        return text, metadata

# Test cleaning
cleaner = TextCleaner(min_length=20, max_length=5000)

for i, doc in enumerate(sample_dataset):
    cleaned, meta = cleaner.clean(doc)
    status = "✅ KEPT" if not meta.get('rejected') else f"❌ REJECTED ({meta['reason']})"
    print(f"\nDoc {i+1}: {status}")
    print(f"  Quality: {meta.get('quality_score', 0):.2f}")
    if cleaned:
        print(f"  Length: {meta['original_length']} → {meta['clean_length']}")
        print(f"  Preview: {cleaned[:80]}...")

## 3. Deduplication

In [ ]:
class Deduplicator:
    """Remove duplicate and near-duplicate documents"""
    
    def __init__(self, method='exact', threshold=0.9):
        self.method = method
        self.threshold = threshold
        self.seen_hashes = set()
    
    def exact_hash(self, text: str) -> str:
        """MD5 hash of normalized text"""
        normalized = re.sub(r'\s+', ' ', text.lower().strip())
        return hashlib.md5(normalized.encode()).hexdigest()
    
    def minhash(self, text: str, num_hashes=5, k=3) -> List[str]:
        """Simple MinHash for near-duplicate detection"""
        # Generate k-shingles
        text = text.lower()
        shingles = [text[i:i+k] for i in range(len(text) - k + 1)]
        
        # Simple hash signatures
        signatures = []
        for seed in range(num_hashes):
            min_hash = float('inf')
            for shingle in shingles:
                h = int(hashlib.md5((shingle + str(seed)).encode()).hexdigest(), 16)
                min_hash = min(min_hash, h)
            signatures.append(str(min_hash))
        
        return signatures
    
    def jaccard_similarity(self, sig1: List[str], sig2: List[str]) -> float:
        """Calculate Jaccard similarity of signatures"""
        set1, set2 = set(sig1), set(sig2)
        intersection = len(set1 & set2)
        union = len(set1 | set2)
        return intersection / union if union > 0 else 0.0
    
    def deduplicate(self, documents: List[str]) -> Tuple[List[str], Dict]:
        """Remove duplicates from document list"""
        unique_docs = []
        stats = {'total': len(documents), 'exact_dupes': 0, 'near_dupes': 0, 'kept': 0}
        
        for doc in documents:
            # Exact deduplication
            doc_hash = self.exact_hash(doc)
            if doc_hash in self.seen_hashes:
                stats['exact_dupes'] += 1
                continue
            
            # Near-duplicate detection
            if self.method == 'minhash':
                current_sig = self.minhash(doc)
                is_duplicate = False
                for seen_doc in unique_docs:
                    seen_sig = self.minhash(seen_doc)
                    sim = self.jaccard_similarity(current_sig, seen_sig)
                    if sim >= self.threshold:
                        is_duplicate = True
                        stats['near_dupes'] += 1
                        break
                if is_duplicate:
                    continue
            
            self.seen_hashes.add(doc_hash)
            unique_docs.append(doc)
            stats['kept'] += 1
        
        return unique_docs, stats

# Clean all documents first
cleaned_docs = []
for doc in sample_dataset:
    cleaned, meta = cleaner.clean(doc)
    if cleaned:
        cleaned_docs.append(cleaned)

print(f"After cleaning: {len(cleaned_docs)} documents")

# Deduplicate
dedup = Deduplicator(method='exact')
unique_docs, stats = dedup.deduplicate(cleaned_docs)

print(f"\n📊 Deduplication Results:")
print(f"   Total input: {stats['total']}")
print(f"   Exact duplicates removed: {stats['exact_dupes']}")
print(f"   Documents kept: {stats['kept']}")
print(f"   Reduction: {(1 - stats['kept']/stats['total'])*100:.1f}%")

## 4. Tokenization & Chunking

In [ ]:
class SimpleTokenizer:
    """Simple character-level tokenizer for demonstration"""
    
    def __init__(self):
        self.vocab = {'<pad>': 0, '<unk>': 1, '<s>': 2, '</s>': 3}
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}
    
    def build_vocab(self, texts: List[str]):
        """Build vocabulary from texts"""
        chars = set()
        for text in texts:
            chars.update(text)
        
        for char in sorted(chars):
            if char not in self.vocab:
                idx = len(self.vocab)
                self.vocab[char] = idx
                self.inverse_vocab[idx] = char
        
        return self
    
    def encode(self, text: str) -> List[int]:
        """Text to token IDs"""
        return [self.vocab.get(c, self.vocab['<unk>']) for c in text]
    
    def decode(self, ids: List[int]) -> str:
        """Token IDs to text"""
        return ''.join(self.inverse_vocab.get(i, '<unk>') for i in ids)
    
    def vocab_size(self) -> int:
        return len(self.vocab)

# Build tokenizer
tokenizer = SimpleTokenizer()
tokenizer.build_vocab(unique_docs)

print(f"Vocabulary size: {tokenizer.vocab_size()}")
print(f"Sample vocab: {list(tokenizer.vocab.items())[:10]}")

In [ ]:
class DataChunker:
    """Chunk text into fixed-size sequences for training"""
    
    def __init__(self, seq_length=128, stride=64):
        self.seq_length = seq_length
        self.stride = stride
    
    def chunk_text(self, text: str, tokenizer) -> List[Dict]:
        """Create overlapping chunks from text"""
        tokens = tokenizer.encode(text)
        chunks = []
        
        for i in range(0, len(tokens) - self.seq_length + 1, self.stride):
            chunk_tokens = tokens[i:i + self.seq_length]
            
            # Pad if necessary
            if len(chunk_tokens) < self.seq_length:
                chunk_tokens += [tokenizer.vocab['<pad>']] * (self.seq_length - len(chunk_tokens))
            
            chunks.append({
                'input_ids': chunk_tokens[:-1],
                'labels': chunk_tokens[1:],  # Next token prediction
                'length': len([t for t in chunk_tokens if t != tokenizer.vocab['<pad>']])
            })
        
        return chunks
    
    def create_dataset(self, documents: List[str], tokenizer) -> List[Dict]:
        """Create full training dataset"""
        all_chunks = []
        
        for doc in documents:
            chunks = self.chunk_text(doc, tokenizer)
            all_chunks.extend(chunks)
        
        return all_chunks

# Create chunks
chunker = DataChunker(seq_length=64, stride=32)
train_data = chunker.create_dataset(unique_docs, tokenizer)

print(f"Total training chunks: {len(train_data)}")
print(f"Sequence length: {len(train_data[0]['input_ids'])}")
print(f"\nSample chunk:")
print(f"  Input: {tokenizer.decode(train_data[0]['input_ids'][:50])}...")
print(f"  Label: {tokenizer.decode(train_data[0]['labels'][:50])}...")

## 5. DataLoader & Batching

In [ ]:
class SimpleDataLoader:
    """Simple DataLoader for LLM training"""
    
    def __init__(self, data: List[Dict], batch_size=4, shuffle=True):
        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = list(range(len(data)))
        if shuffle:
            random.shuffle(self.indices)
    
    def __iter__(self) -> Iterator[Dict]:
        """Iterate over batches"""
        batch = []
        for idx in self.indices:
            batch.append(self.data[idx])
            if len(batch) == self.batch_size:
                yield self._collate(batch)
                batch = []
        
        # Final batch (drop last if incomplete)
        if batch and len(batch) == self.batch_size:
            yield self._collate(batch)
    
    def _collate(self, batch: List[Dict]) -> Dict:
        """Stack batch items into tensors"""
        input_ids = np.array([item['input_ids'] for item in batch])
        labels = np.array([item['labels'] for item in batch])
        
        return {
            'input_ids': input_ids,
            'labels': labels,
            'batch_size': len(batch)
        }
    
    def __len__(self) -> int:
        return len(self.data) // self.batch_size

# Create DataLoader
dataloader = SimpleDataLoader(train_data, batch_size=2, shuffle=True)

print(f"Number of batches: {len(dataloader)}")
print(f"\nFirst batch shape:")

for batch in dataloader:
    print(f"  input_ids: {batch['input_ids'].shape}")
    print(f"  labels: {batch['labels'].shape}")
    print(f"  Sample input: {batch['input_ids'][0][:20]}")
    break

## 6. Full Pipeline Visualization

In [ ]:
# Pipeline statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Document counts at each stage
stages = ['Raw', 'After\nCleaning', 'After\nDeduplication', 'Chunks']
counts = [len(sample_dataset), len(cleaned_docs), len(unique_docs), len(train_data)]
colors = ['#FF6B6B', '#FFA07A', '#4ECDC4', '#45B7D1']

bars = axes[0, 0].bar(stages, counts, color=colors, edgecolor='black')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Documents at Each Pipeline Stage')
axes[0, 0].set_yscale('log')
for bar, count in zip(bars, counts):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                   str(count), ha='center', fontweight='bold')

# 2. Quality score distribution
quality_scores = []
for doc in sample_dataset:
    _, meta = cleaner.clean(doc)
    quality_scores.append(meta.get('quality_score', 0))

axes[0, 1].hist(quality_scores, bins=10, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(0.5, color='red', linestyle='--', label='Threshold')
axes[0, 1].set_xlabel('Quality Score')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Text Quality Distribution')
axes[0, 1].legend()

# 3. Chunk length distribution
lengths = [chunk['length'] for chunk in train_data]
axes[1, 0].hist(lengths, bins=15, color='#45B7D1', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Sequence Length (non-pad tokens)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Training Sequence Length Distribution')

# 4. Vocabulary frequency
all_tokens = []
for chunk in train_data:
    all_tokens.extend(chunk['input_ids'])

token_counts = Counter(all_tokens)
top_tokens = token_counts.most_common(15)
tokens, counts = zip(*top_tokens)
token_labels = [tokenizer.inverse_vocab[t] for t in tokens]

axes[1, 1].barh(token_labels[::-1], counts[::-1], color='#FF6B6B', edgecolor='black')
axes[1, 1].set_xlabel('Frequency')
axes[1, 1].set_title('Top 15 Most Frequent Tokens')

plt.tight_layout()
plt.show()

## 7. Save Processed Data

In [ ]:
# Save processed dataset
output = {
    'vocab': tokenizer.vocab,
    'train_data': train_data,
    'metadata': {
        'num_documents': len(unique_docs),
        'num_chunks': len(train_data),
        'seq_length': chunker.seq_length,
        'vocab_size': tokenizer.vocab_size(),
        'total_tokens': sum(len(chunk['input_ids']) for chunk in train_data)
    }
}

# Save as JSON
with open('processed_data.json', 'w') as f:
    json.dump(output, f, indent=2)

print("💾 Saved: processed_data.json")
print(f"\n📊 Dataset Statistics:")
for key, val in output['metadata'].items():
    print(f"   {key}: {val:,}")

## 🎯 Exercises

1. **Scale Up**: Apply this pipeline to a real dataset (Wikipedia dump, Common Crawl sample).
2. **Streaming**: Modify DataLoader to stream from disk for datasets larger than RAM.
3. **Multilingual**: Add language detection and filtering for multilingual data.
4. **Quality Classifier**: Train a simple classifier to predict text quality instead of heuristics.
5. **Hugging Face Integration**: Use `datasets` library for efficient processing.
6. **Memory Map**: Save processed data in memory-mapped format for faster loading.